# 07 Product Problem Prioritisation
E-Commerce Product Analytics ? Opportunity Prioritization
All metrics re-queried from data/ecommerce_analytics.duckdb


In [ ]:
import sys, os, duckdb, pandas as pd, math
from scipy import stats
from IPython.display import Image, display
sys.path.append('../src')
con = duckdb.connect('../data/ecommerce_analytics.duckdb', read_only=True)
def q(sql): return con.execute(sql).df()
print("Connected.")


## 1. Funnel Baseline


In [ ]:
f = q('''SELECT COUNT(DISTINCT s.session_id) AS sessions,
       COUNT(DISTINCT pv.session_id) AS pdp_sessions,
       COUNT(DISTINCT ce.session_id) AS cart_sessions,
       COUNT(DISTINCT o.session_id)  AS orders
FROM sessions s
LEFT JOIN product_views pv ON s.session_id = pv.session_id
LEFT JOIN cart_events   ce ON s.session_id = ce.session_id
LEFT JOIN orders        o  ON s.session_id = o.session_id''')
print(f.to_string())


## 2. Problem A - Search


In [ ]:
df_a = q('''WITH td AS (SELECT ARRAY_LENGTH(STRING_SPLIT(TRIM(query_text),' ')) AS tok,
           is_zero_result, has_pdp_click, session_id, user_id FROM search_events)
SELECT CASE WHEN tok>=4 THEN '4+ tokens' ELSE '<4 tokens' END AS tier,
    COUNT(*) AS searches, COUNT(DISTINCT session_id) AS sessions,
    ROUND(100.0*SUM(CASE WHEN is_zero_result THEN 1 ELSE 0 END)/COUNT(*),2) AS zrr_pct,
    ROUND(100.0*SUM(CASE WHEN has_pdp_click THEN 1 ELSE 0 END)/COUNT(*),2) AS ctr_pct
FROM td GROUP BY 1''')
display(df_a)


## 3. Problem B - PDP Stockout


In [ ]:
df_b = q("SELECT is_size_in_stock, COUNT(*) AS views, ROUND(100.0*SUM(CASE WHEN added_to_cart THEN 1 ELSE 0 END)/COUNT(*),4) AS atcr FROM product_views GROUP BY 1 ORDER BY 1 DESC")
display(df_b)


## 4. Problem C - Shipping Cliff


In [ ]:
df_c = q('''WITH cs AS (SELECT ce.session_id, SUM(ce.item_price*ce.quantity) AS cart_gmv,
           MAX(CASE WHEN o.order_id IS NOT NULL THEN 1 ELSE 0 END) AS ordered
    FROM cart_events ce LEFT JOIN orders o ON ce.session_id = o.session_id GROUP BY 1)
SELECT CASE WHEN cart_gmv<25 THEN 'a.<$25' WHEN cart_gmv<38 THEN 'b.$25-$37'
     WHEN cart_gmv<50 THEN 'c.$38-$49 (CLIFF)' WHEN cart_gmv<75 THEN 'd.$50-$74 (FREE)'
     ELSE 'e.$75+' END AS tier, COUNT(*) AS carts, SUM(ordered) AS orders,
    ROUND(100.0*SUM(ordered)/COUNT(*),2) AS cto_pct FROM cs GROUP BY 1 ORDER BY 1''')
display(df_c)


## 5. Problem D - Mobile Web


In [ ]:
df_d = q('''SELECT s.platform, COUNT(DISTINCT ce.session_id) AS carts,
    COUNT(DISTINCT o.session_id) AS orders,
    ROUND(100.0*COUNT(DISTINCT o.session_id)/NULLIF(COUNT(DISTINCT ce.session_id),0),2) AS cto_pct
FROM sessions s JOIN cart_events ce ON s.session_id=ce.session_id
LEFT JOIN orders o ON s.session_id=o.session_id GROUP BY 1 ORDER BY cto_pct DESC''')
display(df_d)


## 6. Prioritisation Scores


In [ ]:
prio = pd.DataFrame([
    {'Problem':'A - Search Discovery',    'Reach':4,'Impact':4,'Confidence':5,'Effort':5},
    {'Problem':'D - Mobile Web Checkout', 'Reach':3,'Impact':4,'Confidence':5,'Effort':4},
    {'Problem':'C - Shipping Cliff',      'Reach':2,'Impact':3,'Confidence':5,'Effort':2},
    {'Problem':'B - PDP Stockout',        'Reach':2,'Impact':4,'Confidence':5,'Effort':3},
])
prio['Priority_Score'] = (prio['Reach']*prio['Impact']*prio['Confidence']/prio['Effort']).round(1)
prio['Evidence_Score'] = prio['Reach']*prio['Impact']*prio['Confidence']
display(prio.sort_values('Priority_Score', ascending=False))


## 7. Visualisations


In [ ]:
display(Image('../reports/figures/13_problem_prioritization_matrix.png'))
display(Image('../reports/figures/14_impact_vs_effort.png'))
display(Image('../reports/figures/15_problem_opportunity_comparison.png'))


## 8. Conclusion
Primary Problem: A - Search Discovery Failure on Specific Queries
Secondary Problem: D - Mobile Web Checkout Friction


In [ ]:
con.close()
print("Opportunity prioritization notebook complete.")
